<p style="text-align:center"> 
    <a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/" target="_blank"> 
    <img src="../assets/logo.png" width="200" alt="Flavio Aguirre Logo"> 
    </a>
</p>

<h1 align="center"><strong>Weather Wise – 01 · Data Collection</strong></h1>
<hr>

In this notebook, we perform the **first step of the pipeline**: **downloading the raw weather dataset** from the original source, performing a **basic security check**, and storing a **frozen copy** in `data/raw/` for use in the rest of the project.
The objectives of this notebook are:
- To verify that the data can be loaded correctly from the remote URL.
- Verify that the data can be loaded correctly from the remote URL.
- Inspect the shape, columns, and basic statistics of the dataset.
- Confirm the presence of the target variable (`RainTomorrow`).
- Save a local copy of the raw data to ensure reproducibility.

In [12]:
## Let's start by importing the necessary libraries

import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

## 1. Load the data

We load the dataset from the public URL provided by IBM/Coursera.  
This dataset contains **daily weather observations in Australia** (2008–2017), which we will later use to predict whether it will rain tomorrow.

In [13]:
## Load the data

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/_0eYOqji3unP1tDNKWZMjg/weatherAUS-2.csv"
df = pd.read_csv(url)

print("Data loaded successfully from URL")
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

df.head()

Data loaded successfully from URL
Dataset shape: 145460 rows × 23 columns


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


## 2. Basic sanity checks

Before doing any serious analysis or modeling,  
we run quick checks to understand the **completeness** and **structure** of the data.

In [14]:
## Check non-null counts per column

df.count()

Date             145460
Location         145460
MinTemp          143975
MaxTemp          144199
Rainfall         142199
Evaporation       82670
Sunshine          75625
WindGustDir      135134
WindGustSpeed    135197
WindDir9am       134894
WindDir3pm       141232
WindSpeed9am     143693
WindSpeed3pm     142398
Humidity9am      142806
Humidity3pm      140953
Pressure9am      130395
Pressure3pm      130432
Cloud9am          89572
Cloud3pm          86102
Temp9am          143693
Temp3pm          141851
RainToday        142199
RainTomorrow     142193
dtype: int64

Columns like `Sunshine` and `Evaporation` are intuitively important for a weather model,  
but they show a **high number of missing values**, which will require careful treatment later  
(imputation strategies or potential removal, depending on their impact on the model).

### 2.1. Descriptive statistics

Basic descriptive statistics for numerical columns help us spot obvious issues such as:

- missing values,
- extreme outliers,
- columns with no variance.

In [15]:
## Statistics

df.describe().T

,count,mean,std,min,25%,50%,75%,max
MinTemp,143975.0,12.194034,6.398495,-8.5,7.6,12.0,16.9,33.9
MaxTemp,144199.0,23.221348,7.119049,-4.8,17.9,22.6,28.2,48.1
Rainfall,142199.0,2.360918,8.478060,0.0,0.0,0.0,0.8,371.0
Evaporation,82670.0,5.468232,4.193704,0.0,2.6,4.8,7.4,145.0
Sunshine,75625.0,7.611178,3.785483,0.0,4.8,8.4,10.6,14.5
WindGustSpeed,135197.0,40.035230,13.607062,6.0,31.0,39.0,48.0,135.0
WindSpeed9am,143693.0,14.043426,8.915375,0.0,7.0,13.0,19.0,130.0
WindSpeed3pm,142398.0,18.662657,8.809800,0.0,13.0,19.0,24.0,87.0
Humidity9am,142806.0,68.880831,19.029164,0.0,57.0,70.0,83.0,100.0
Humidity3pm,140953.0,51.539116,20.795902,0.0,37.0,52.0,66.0,100.0


### 2.2. Data types

We also need to confirm that:

- Numerical features are stored as numeric types (e.g. `float64`),
- Categorical features (e.g. `Location`, `WindDir3pm`) are stored as objects and will later be encoded,
- The target variable `RainTomorrow` is present and correctly typed.

In [16]:
## DataTypes

df.dtypes

Date              object
Location          object
MinTemp          float64
MaxTemp          float64
Rainfall         float64
Evaporation      float64
Sunshine         float64
WindGustDir       object
WindGustSpeed    float64
WindDir9am        object
WindDir3pm        object
WindSpeed9am     float64
WindSpeed3pm     float64
Humidity9am      float64
Humidity3pm      float64
Pressure9am      float64
Pressure3pm      float64
Cloud9am         float64
Cloud3pm         float64
Temp9am          float64
Temp3pm          float64
RainToday         object
RainTomorrow      object
dtype: object

### 2.3. Additional dataset information

The `.info()` summary provides:

- Total number of non-null entries,
- Memory usage,
- A quick view of which columns have missing values.

In [17]:
## More info about dataset

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  object 
 1   Location       145460 non-null  object 
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object 
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object 
 10  WindDir3pm     141232 non-null  object 
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null

## 3. Minimal target and integrity checks

For this project, the **target variable** is `RainTomorrow`.  
We perform a simple check to verify that:

- The column exists,
- It contains at least some non-null values.

In [18]:
# Basic target checks
assert "RainTomorrow" in df.columns, "Target column 'RainTomorrow' is missing from the dataset."

target_non_null = df["RainTomorrow"].notnull().sum()
print(f"'RainTomorrow' non-null values: {target_non_null} / {len(df)}")

df["RainTomorrow"].value_counts(dropna=False)

'RainTomorrow' non-null values: 142193 / 145460


RainTomorrow
No     110316
Yes     31877
NaN      3267
Name: count, dtype: int64

If the assertions above pass and the value counts look reasonable, we can safely **store this snapshot as our raw dataset**.

This ensures that downstream notebooks (EDA, preprocessing, modeling) always start from the **same immutable raw file**, which is an important practice for reproducibility.

In [19]:
### We proceed to save this initial dataset

raw_data_dir = "../data/raw/"
os.makedirs(raw_data_dir, exist_ok=True)

output_path = os.path.join(raw_data_dir, "weatherAUS-data.csv")
df.to_csv(output_path, index=False)

print(f"Raw dataset saved to: {output_path}")

Raw dataset saved to: ../data/raw/weatherAUS-data.csv


<br>

<hr>

## Author

<a href="https://www.linkedin.com/in/flavio-aguirre-12784a252/">**Flavio Aguirre**</a>  
<a href="https://coursera.org/share/e27ae5af81b56f99a2aa85289b7cdd04">***Data Scientist***</a>